# LeetCode 782 — Transform to Chessboard bằng Python

- Link: https://leetcode.com/problems/transform-to-chessboard/
- Độ khó: **Hard**
- Chủ đề: Ma trận, Toán (bất biến), Tham lam đếm vị trí sai, Bitmask

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def movesToChessboard(self, board: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho một bảng `n × n` gồm 0 và 1. Mỗi nước đi được **đổi chỗ hai hàng bất kỳ** hoặc **hai cột bất kỳ**. Hỏi cần ít nhất bao nhiêu nước để biến bảng thành **bàn cờ vua** (không có hai ô kề cạnh nào giống nhau), hoặc `-1` nếu không thể.

Bài này khó vì không gian trạng thái rất lớn, nhưng lời giải cuối cùng chỉ là **vài phép đếm `O(n²)`**. Chìa khoá là tìm ra các **bất biến** (thứ không đổi khi đổi hàng/cột) và nhận ra rằng **hàng và cột độc lập với nhau**.

Sau notebook này bạn sẽ:

1. Viết **BFS trên trạng thái bảng** làm lời giải chuẩn cho `n` nhỏ.
2. Tìm **bất biến** của phép đổi hàng/cột và dùng nó để kiểm tra "có thể hay không".
3. Hiểu vì sao bài **tách thành hai bài 1 chiều độc lập** (sắp hàng, sắp cột).
4. Tính số lần đổi chỗ tối thiểu để biến một dãy 0/1 thành dãy xen kẽ bằng cách **đếm vị trí sai / 2**.
5. Xử lý đúng chẵn/lẻ của `n` — nguồn lỗi phổ biến nhất của bài.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho ma trận nhị phân `board` kích thước `n × n`. Một nước đi: đổi chỗ hai hàng bất kỳ, hoặc đổi chỗ hai cột bất kỳ. Trả về số nước đi ít nhất để `board` thành **bàn cờ** — tức là không có hai ô chung cạnh nào có cùng giá trị (4 hướng). Nếu không thể, trả về `-1`.

Có đúng hai bàn cờ `n × n`: một bàn bắt đầu bằng `0` ở góc trái trên, một bàn bắt đầu bằng `1`.

**Ví dụ 1**

```text
Input: board = [[0,1,1,0],[0,1,1,0],[1,0,0,1],[1,0,0,1]]
Output: 2
Giải thích: đổi cột 0 với cột 1 → [[1,0,1,0],[1,0,1,0],[0,1,0,1],[0,1,0,1]],
            rồi đổi hàng 1 với hàng 2 → bàn cờ.
```

**Ví dụ 2**

```text
Input: board = [[0,1],[1,0]]
Output: 0
Giải thích: đã là bàn cờ.
```

**Ví dụ 3**

```text
Input: board = [[1,0],[1,0]]
Output: -1
Giải thích: dù đổi thế nào cũng không thành bàn cờ.
```

**Ràng buộc**

- `n == board.length == board[i].length`
- `2 <= n <= 30`
- `board[i][j]` là `0` hoặc `1`.

In [1]:
from typing import List

EXAMPLES = [
    ([[0, 1, 1, 0], [0, 1, 1, 0], [1, 0, 0, 1], [1, 0, 0, 1]], 2),
    ([[0, 1], [1, 0]], 0),
    ([[1, 0], [1, 0]], -1),
]

def show(board):
    for row in board:
        print(' '.join('█' if x else '·' for x in row))

for b, exp in EXAMPLES:
    show(b)
    print("kỳ vọng:", exp)
    print()

· █ █ ·
· █ █ ·
█ · · █
█ · · █
kỳ vọng: 2

· █
█ ·
kỳ vọng: 0

█ ·
█ ·
kỳ vọng: -1



## 3. Kiến thức nền

### 3.1. Đổi chỗ hai hàng / hai cột trong Python

Đổi hàng: `b[i], b[j] = b[j], b[i]`. Đổi cột: phải đổi từng phần tử trong mỗi hàng.

In [2]:
import copy

def swap_rows(b, i, j):
    b = copy.deepcopy(b)
    b[i], b[j] = b[j], b[i]
    return b

def swap_cols(b, i, j):
    b = copy.deepcopy(b)
    for row in b:
        row[i], row[j] = row[j], row[i]
    return b

b0 = EXAMPLES[0][0]
show(swap_cols(b0, 0, 1)); print()
show(swap_rows(swap_cols(b0, 0, 1), 1, 2))

█ · █ ·
█ · █ ·
· █ · █
· █ · █

█ · █ ·
· █ · █
█ · █ ·
· █ · █


### 3.2. Kiểm tra một bảng có phải bàn cờ

Bàn cờ ⇔ `board[i][j] == (i + j + board[0][0]) % 2` với mọi `i, j`.

In [3]:
def is_chessboard(b):
    n = len(b)
    return all(b[i][j] == (i + j + b[0][0]) % 2 for i in range(n) for j in range(n))

print(is_chessboard([[0, 1], [1, 0]]), is_chessboard([[1, 0], [1, 0]]))

True False


### 3.3. Phép XOR `^` và phần bù của một hàng

`a ^ b` = 1 nếu `a ≠ b`. Hàng `r` và hàng "bù" `[1 - x for x in r]` khác nhau ở **mọi** vị trí. Trong bàn cờ, mọi hàng hoặc bằng hàng 0, hoặc là phần bù của hàng 0.

In [4]:
r = [0, 1, 1, 0]
comp = [1 - x for x in r]
print(comp, [x ^ y for x, y in zip(r, comp)])

[1, 0, 0, 1] [1, 1, 1, 1]


### 3.4. Số lần đổi chỗ tối thiểu để sắp một dãy về một dãy đích (hai loại phần tử)

Nếu dãy chỉ có **hai loại** phần tử và ta muốn biến nó thành dãy đích có **cùng số lượng mỗi loại**, thì số lần đổi chỗ tối thiểu = (số vị trí sai) / 2. Lý do: các vị trí sai gồm `k` chỗ "đáng là A nhưng là B" và đúng `k` chỗ "đáng là B nhưng là A"; mỗi lần đổi sửa được tối đa 2 chỗ sai.

In [5]:
def min_swaps_two_types(seq, target):
    assert sorted(seq) == sorted(target)
    wrong = sum(a != b for a, b in zip(seq, target))
    return wrong // 2

print(min_swaps_two_types([1, 1, 0, 0], [0, 1, 0, 1]))   # sai ở vị trí 0 và 3 -> 1 lần
print(min_swaps_two_types([1, 1, 0, 0], [1, 0, 1, 0]))   # sai ở vị trí 1 và 2 -> 1 lần

1
1


### 3.5. Số lần đổi chỗ tối thiểu để thực hiện một hoán vị = `n - số chu trình`

Với hoán vị tổng quát (các phần tử phân biệt), số lần đổi chỗ ít nhất để đưa về đúng vị trí là `n - (số chu trình)`. Ta sẽ dùng công thức này trong lời giải "liệt kê hoán vị".

In [6]:
def swaps_for_perm(p):
    n = len(p)
    seen = [False] * n
    cycles = 0
    for i in range(n):
        if not seen[i]:
            cycles += 1
            j = i
            while not seen[j]:
                seen[j] = True
                j = p[j]
    return n - cycles

print(swaps_for_perm([0, 1, 2]), swaps_for_perm([1, 0, 2]), swaps_for_perm([1, 2, 0]))

0 1 2


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đổi hàng/cột giữ nguyên những gì?

- Đổi **cột** không làm thay đổi **tập các hàng** về mặt "hàng nào bằng hàng nào": nếu hai hàng giống nhau trước khi đổi cột thì vẫn giống nhau sau đó (vì cả hai bị hoán vị cột theo cùng một cách). Tương tự với hàng bù.
- Đổi **hàng** không làm thay đổi nội dung từng hàng, chỉ đổi thứ tự.

→ **Bất biến 1:** "hàng `i` bằng hàng `0`" hay "hàng `i` là phần bù của hàng `0`" không đổi qua mọi nước đi. Trong bàn cờ, **mọi** hàng thuộc một trong hai loại đó. Vậy bảng ban đầu cũng phải như vậy, nếu không → `-1`.

→ **Bất biến 2:** số lượng số 1 trong mỗi hàng không đổi khi đổi hàng; khi đổi cột cũng không đổi. Trong bàn cờ, mỗi hàng có `n/2` số 1 (n chẵn) hoặc `⌊n/2⌋`/`⌈n/2⌉` (n lẻ).

→ Tương tự cho cột.

### 4.2. Kiểm tra bằng 4 góc hình chữ nhật

Điều kiện "mọi hàng bằng hàng 0 hoặc bù hàng 0" tương đương với: với mọi `i, j`,

```text
board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] == 0
```

(bốn góc của mọi hình chữ nhật có một góc ở `(0,0)` có số lượng 1 chẵn).

### 4.3. Hàng và cột độc lập

Đổi hàng chỉ thay đổi **thứ tự các hàng**, đổi cột chỉ thay đổi **thứ tự các cột**. Bảng cuối = (hoán vị hàng) ∘ (hoán vị cột) áp lên bảng ban đầu, bất kể thứ tự thực hiện. Vậy:

```text
tổng số nước = (số lần đổi hàng tối thiểu để các hàng xen kẽ) + (số lần đổi cột tối thiểu để các cột xen kẽ)
```

### 4.4. Tính tay ví dụ 1

```text
0 1 1 0      hàng 0 = A = 0110
0 1 1 0      hàng 1 = A
1 0 0 1      hàng 2 = B = 1001 (bù của A)
1 0 0 1      hàng 3 = B
```

- Hàng: dãy loại `A A B B` cần thành `A B A B` hoặc `B A B A`. So với `ABAB`: sai ở vị trí 1, 2 → 1 lần đổi. So với `BABA`: sai ở 0, 3 → 1 lần. → **1**.
- Cột: cột 0 = `0011`, cột 1 = `1100` (bù), cột 2 = `1100`, cột 3 = `0011`. Dãy loại cột `C D D C` → thành `CDCD` (sai 2, 3 → 1 lần) hoặc `DCDC` (sai 0,1 → 1 lần) → **1**.
- Tổng: **2** ✔.

In [7]:
b = EXAMPLES[0][0]
n = len(b)
row_type = ['A' if b[i] == b[0] else ('B' if all(b[i][j] != b[0][j] for j in range(n)) else '?') for i in range(n)]
cols = [[b[i][j] for i in range(n)] for j in range(n)]
col_type = ['C' if cols[j] == cols[0] else ('D' if all(cols[j][i] != cols[0][i] for i in range(n)) else '?') for j in range(n)]
print("loại hàng:", row_type)
print("loại cột :", col_type)

loại hàng: ['A', 'A', 'B', 'B']
loại cột : ['C', 'D', 'D', 'C']


## 5. Cách 1 — Brute Force: BFS trên trạng thái bảng

**Ý tưởng.** Mỗi bảng là một đỉnh; mỗi nước đổi hàng/cột là một cạnh. BFS từ bảng ban đầu, lần đầu gặp một bàn cờ thì trả số bước. Nếu duyệt hết mà không gặp → `-1`.

Số trạng thái tới được ≤ `(n!)²` (hoán vị hàng × hoán vị cột): với `n = 4` là 576, `n = 5` là 14 400 — đủ nhỏ để làm **chuẩn đối chiếu**. Lời giải này **không** dùng bất kỳ suy luận nào, nên rất đáng tin.

In [8]:
from collections import deque

def to_key(b):
    return tuple(tuple(r) for r in b)

def moves_bfs(board):
    n = len(board)
    start = to_key(board)
    dist = {start: 0}
    q = deque([start])
    while q:
        s = q.popleft()
        if is_chessboard(s):
            return dist[s]
        rows = list(s)
        for i in range(n):
            for j in range(i + 1, n):
                # đổi hàng i, j
                r2 = rows[:]
                r2[i], r2[j] = r2[j], r2[i]
                t = tuple(r2)
                if t not in dist:
                    dist[t] = dist[s] + 1
                    q.append(t)
                # đổi cột i, j
                t = tuple(tuple(r[j] if k == i else r[i] if k == j else r[k] for k in range(n)) for r in rows)
                if t not in dist:
                    dist[t] = dist[s] + 1
                    q.append(t)
    return -1

for b, exp in EXAMPLES:
    print(moves_bfs(b), "(kỳ vọng", exp, ")")
    assert moves_bfs(b) == exp

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


**Độ phức tạp.** `O((n!)² · n² · n²)` (số trạng thái × số nước × chi phí tạo bảng mới). Với `n = 30` là không tưởng.

**Nút thắt.** BFS xét hàng và cột **đan xen**, trong khi thực ra chúng độc lập. Tách ra sẽ giảm từ `(n!)²` xuống `2 · n!`, rồi các bất biến giảm tiếp xuống `O(n²)`.

## 6. Bước đột phá

**Insight 1 — Khả thi ⇔ hai điều kiện bất biến.**
- *Nếu* tồn tại `i, j` với `board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] == 1` *thì* `-1`.
- *Nếu* số 1 trong hàng 0 (hoặc cột 0) không nằm trong `[n//2, (n+1)//2]` *thì* `-1`. (Khi mọi hàng là A hoặc bù A, điều kiện này trên cột 0 đảm bảo số hàng loại A và loại B cân bằng.)

**Insight 2 — Tách hàng/cột.** Đáp án = `swaps(hàng) + swaps(cột)`.

**Insight 3 — Mỗi chiều là bài 1D "biến dãy 0/1 thành xen kẽ".** Loại của hàng `i` được nhận diện bởi `board[i][0]` (vì hàng A và hàng B khác nhau ở cột 0). Vậy sắp hàng ⇔ biến **cột 0** thành dãy xen kẽ. Tương tự sắp cột ⇔ biến **hàng 0** thành dãy xen kẽ.

**Insight 4 — Đếm sai rồi chia 2, cẩn thận chẵn/lẻ.** Gọi `mis` = số vị trí `i` mà `board[i][0] == i % 2` (sai so với mẫu `1,0,1,0,...`).
- *Nếu* `n` chẵn: hai mẫu `1010...` và `0101...` đều hợp lệ; số sai so với mẫu kia là `n - mis`. Chọn `min(mis, n - mis) / 2`.
- *Nếu* `n` lẻ: chỉ **một** mẫu có số lượng 1 đúng; mẫu đó cho số sai **chẵn**. Nếu `mis` lẻ thì dùng `n - mis`. Rồi chia 2.

In [9]:
def mismatch(seq):
    """Số vị trí i mà seq[i] == i % 2, tức sai so với mẫu 1,0,1,0,..."""
    return sum(seq[i] == i % 2 for i in range(len(seq)))

for seq in ([0, 0, 1, 1], [1, 1, 0, 0], [0, 1, 1], [1, 1, 0], [1, 0, 1, 0, 0]):
    m = mismatch(seq)
    print(seq, "sai so với 1010..:", m, "| sai so với 0101..:", len(seq) - m)

[0, 0, 1, 1] sai so với 1010..: 2 | sai so với 0101..: 2
[1, 1, 0, 0] sai so với 1010..: 2 | sai so với 0101..: 2
[0, 1, 1] sai so với 1010..: 2 | sai so với 0101..: 1
[1, 1, 0] sai so với 1010..: 2 | sai so với 0101..: 1
[1, 0, 1, 0, 0] sai so với 1010..: 1 | sai so với 0101..: 4


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Liệt kê hoán vị hàng và hoán vị cột (dùng tính độc lập)

Vì bảng cuối = (hoán vị hàng `P`) ∘ (hoán vị cột `Q`) áp lên bảng ban đầu, và chi phí thực hiện một hoán vị bằng `n - số chu trình`, ta liệt kê mọi cặp `(P, Q)` cho ra bàn cờ và lấy min của `swaps(P) + swaps(Q)`. Cắt tỉa: sau khi chọn `P`, cột 0 của các hàng phải xen kẽ; khi chọn `Q`, hàng 0 phải xen kẽ. Cách này vẫn giai thừa, chỉ dùng cho `n ≤ 5`, nhưng nó **kiểm chứng** ý tưởng "chi phí = chi phí hàng + chi phí cột".

In [10]:
from itertools import permutations

def moves_perm(board):
    n = len(board)
    best = None
    for P in permutations(range(n)):
        rows = [board[p] for p in P]
        # kiểm tra nhanh: các hàng phải xen kẽ loại theo cột 0
        if any(rows[i][0] == rows[i + 1][0] for i in range(n - 1)):
            continue
        sp = swaps_for_perm(list(P))
        for Q in permutations(range(n)):
            if any(rows[0][Q[j]] == rows[0][Q[j + 1]] for j in range(n - 1)):
                continue
            nb = [[r[q] for q in Q] for r in rows]
            if is_chessboard(nb):
                cost = sp + swaps_for_perm(list(Q))
                if best is None or cost < best:
                    best = cost
    return -1 if best is None else best

for b, exp in EXAMPLES:
    assert moves_perm(b) == exp
print("Liệt kê hoán vị: OK")

Liệt kê hoán vị: OK


Lưu ý `swaps_for_perm(P)` với `P` là "hàng mới thứ `i` lấy từ hàng cũ `P[i]`" — số lần đổi chỗ của một hoán vị và của hoán vị nghịch đảo bằng nhau (cùng cấu trúc chu trình), nên không cần lo chiều.

**Độ phức tạp:** `O((n!)² · n²)` trường hợp xấu, nhưng cắt tỉa mạnh. Chỉ dùng để đối chiếu.

### 7.2. Cách 3 — Kiểm tra bất biến + đếm sai (lời giải chuẩn `O(n²)`)

Các bước:

1. Kiểm tra mọi `i, j`: `board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] == 0`, nếu không → `-1`.
2. `row_sum = sum(board[0])`, `col_sum = sum(board[i][0])`. Nếu không thuộc `[n//2, (n+1)//2]` → `-1`.
3. `row_mis = #{i : board[i][0] == i % 2}` (quyết định việc đổi **hàng**), `col_mis = #{j : board[0][j] == j % 2}` (đổi **cột**).
4. Chẵn: lấy `min(x, n - x)`; lẻ: nếu `x` lẻ thì `x = n - x`.
5. Trả `(row_mis + col_mis) // 2`.

In [11]:
def moves_math(board):
    n = len(board)
    for i in range(n):
        for j in range(n):
            if board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j]:
                return -1
    row_sum = sum(board[0])
    col_sum = sum(board[i][0] for i in range(n))
    if not (n // 2 <= row_sum <= (n + 1) // 2):
        return -1
    if not (n // 2 <= col_sum <= (n + 1) // 2):
        return -1
    row_mis = sum(board[i][0] == i % 2 for i in range(n))   # đổi hàng
    col_mis = sum(board[0][j] == j % 2 for j in range(n))   # đổi cột
    if n % 2 == 1:
        if row_mis % 2:
            row_mis = n - row_mis
        if col_mis % 2:
            col_mis = n - col_mis
    else:
        row_mis = min(row_mis, n - row_mis)
        col_mis = min(col_mis, n - col_mis)
    return (row_mis + col_mis) // 2

for b, exp in EXAMPLES:
    assert moves_math(b) == exp
print("Bất biến + đếm sai: OK")

Bất biến + đếm sai: OK


Code nộp LeetCode cho cách 3:

In [12]:
class SolutionMath:
    def movesToChessboard(self, board: List[List[int]]) -> int:
        n = len(board)
        if any(board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j]
               for i in range(n) for j in range(n)):
            return -1
        row_sum = sum(board[0])
        col_sum = sum(board[i][0] for i in range(n))
        if not (n // 2 <= row_sum <= (n + 1) // 2 and n // 2 <= col_sum <= (n + 1) // 2):
            return -1
        row_mis = sum(board[i][0] == i % 2 for i in range(n))
        col_mis = sum(board[0][j] == j % 2 for j in range(n))
        if n % 2:
            if row_mis % 2:
                row_mis = n - row_mis
            if col_mis % 2:
                col_mis = n - col_mis
        else:
            row_mis = min(row_mis, n - row_mis)
            col_mis = min(col_mis, n - col_mis)
        return (row_mis + col_mis) // 2

for b, exp in EXAMPLES:
    assert SolutionMath().movesToChessboard(b) == exp
print("SolutionMath: OK")

SolutionMath: OK


**Độ phức tạp cách 3:** `O(n²)` cho bước kiểm tra, `O(n)` cho phần còn lại. Bộ nhớ `O(1)`.

### 7.3. Cách 4 — Bitmask + `Counter` (viết theo "loại hàng")

Mã hoá mỗi hàng thành một số nguyên `n` bit. Bảng khả thi theo chiều hàng ⇔ có **đúng hai** loại hàng `a`, `b` với `a ^ b == (1 << n) - 1` (bù nhau), số lượng chênh nhau ≤ 1, và `a` có số bit 1 bằng `n//2` hoặc `(n+1)//2`. Làm tương tự cho cột (hoặc dùng chính mặt nạ hàng: các cột được xác định bởi các bit của `a`).

Số lần đổi cho một chiều: với mặt nạ `x` (dãy loại theo từng vị trí) và mẫu xen kẽ `m = 0b...0101`, số vị trí sai = `popcount(x ^ m)`. Cách viết này gọn và nhanh.

In [13]:
from collections import Counter

def popcount(x):
    return bin(x).count('1')

def line_cost(mask, n):
    """Số lần đổi tối thiểu để dãy bit `mask` (n bit) thành xen kẽ, hoặc -1."""
    ones = popcount(mask)
    full = (1 << n) - 1
    alt = sum(1 << i for i in range(0, n, 2))   # 0b...010101: bit 0,2,4.. = 1
    best = float('inf')
    for m in (alt, full ^ alt):
        if popcount(m) == ones:
            best = min(best, popcount(mask ^ m) // 2)
    return -1 if best == float('inf') else best

def moves_bitmask(board):
    n = len(board)
    full = (1 << n) - 1
    total = 0
    for lines in (board, [list(c) for c in zip(*board)]):     # hàng, rồi cột
        masks = Counter(sum(v << j for j, v in enumerate(r)) for r in lines)
        if len(masks) != 2:
            return -1
        (a, ca), (b, cb) = masks.items()
        if a ^ b != full or abs(ca - cb) > 1:
            return -1
        c = line_cost(a, n)          # a mô tả loại của các đường vuông góc
        if c < 0:
            return -1
        total += c
    return total

for b, exp in EXAMPLES:
    assert moves_bitmask(b) == exp
print("Bitmask: OK")

Bitmask: OK


Vì sao `line_cost(a, n)` khi duyệt các **hàng** lại đếm số lần đổi **cột**? Mặt nạ `a` của một hàng cho biết giá trị ở từng cột — tức loại của từng cột. Biến `a` thành xen kẽ = sắp các cột. Lượt thứ hai (duyệt các cột) cho số lần đổi hàng. Tổng hai lượt là đáp án.

Trường hợp `n = 1` không xảy ra (`n ≥ 2`), nên luôn có đúng hai loại hàng nếu khả thi.

In [14]:
class SolutionBitmask:
    def movesToChessboard(self, board: List[List[int]]) -> int:
        n = len(board)
        full = (1 << n) - 1
        alt = int('01' * 16, 2) & full       # bit 0, 2, 4, ... bật
        total = 0
        for lines in (board, list(zip(*board))):
            cnt = Counter(sum(v << j for j, v in enumerate(r)) for r in lines)
            if len(cnt) != 2:
                return -1
            (a, ca), (b, cb) = cnt.items()
            if a ^ b != full or abs(ca - cb) > 1:
                return -1
            ones = bin(a).count('1')
            best = n
            for m in (alt, full ^ alt):
                if bin(m).count('1') == ones:
                    best = min(best, bin(a ^ m).count('1') // 2)
            if best == n:
                return -1
            total += best
        return total

for b, exp in EXAMPLES:
    assert SolutionBitmask().movesToChessboard(b) == exp
print("SolutionBitmask: OK")

SolutionBitmask: OK


**Độ phức tạp cách 4:** `O(n²)` để mã hoá, các phép bit `O(n)` (Python int). Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1 và một bảng `5 × 5` (n lẻ).

In [15]:
def trace_math(board):
    n = len(board)
    show(board)
    bad = [(i, j) for i in range(n) for j in range(n)
           if board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j]]
    print("ô vi phạm điều kiện 4 góc:", bad[:5], "..." if len(bad) > 5 else "")
    if bad:
        return -1
    row_sum = sum(board[0]); col_sum = sum(board[i][0] for i in range(n))
    print(f"row_sum={row_sum}, col_sum={col_sum}, cho phép [{n // 2}, {(n + 1) // 2}]")
    if not (n // 2 <= row_sum <= (n + 1) // 2 and n // 2 <= col_sum <= (n + 1) // 2):
        return -1
    col0 = [board[i][0] for i in range(n)]
    row0 = board[0]
    pat = [1 - i % 2 for i in range(n)]
    row_mis = sum(board[i][0] == i % 2 for i in range(n))
    col_mis = sum(board[0][j] == j % 2 for j in range(n))
    print("cột 0 :", col0, "| mẫu 1010..:", pat, "| row_mis =", row_mis)
    print("hàng 0:", row0, "| mẫu 1010..:", pat, "| col_mis =", col_mis)
    if n % 2:
        if row_mis % 2: row_mis = n - row_mis
        if col_mis % 2: col_mis = n - col_mis
        print("n lẻ -> chọn số sai chẵn:", row_mis, col_mis)
    else:
        row_mis, col_mis = min(row_mis, n - row_mis), min(col_mis, n - col_mis)
        print("n chẵn -> lấy min với n - x:", row_mis, col_mis)
    ans = (row_mis + col_mis) // 2
    print("đáp án =", ans)
    return ans

trace_math(EXAMPLES[0][0])
print()
b5 = [[1, 0, 0, 1, 1],
      [0, 1, 1, 0, 0],
      [1, 0, 0, 1, 1],
      [0, 1, 1, 0, 0],
      [0, 1, 1, 0, 0]]
trace_math(b5)
print("BFS kiểm chứng:", moves_bfs(b5))

· █ █ ·
· █ █ ·
█ · · █
█ · · █
ô vi phạm điều kiện 4 góc: [] 
row_sum=2, col_sum=2, cho phép [2, 2]
cột 0 : [0, 0, 1, 1] | mẫu 1010..: [1, 0, 1, 0] | row_mis = 2
hàng 0: [0, 1, 1, 0] | mẫu 1010..: [1, 0, 1, 0] | col_mis = 2
n chẵn -> lấy min với n - x: 2 2
đáp án = 2

█ · · █ █
· █ █ · ·
█ · · █ █
· █ █ · ·
· █ █ · ·
ô vi phạm điều kiện 4 góc: [] 
row_sum=3, col_sum=2, cho phép [2, 3]
cột 0 : [1, 0, 1, 0, 0] | mẫu 1010..: [1, 0, 1, 0, 1] | row_mis = 1
hàng 0: [1, 0, 0, 1, 1] | mẫu 1010..: [1, 0, 1, 0, 1] | col_mis = 2
n lẻ -> chọn số sai chẵn: 4 2
đáp án = 3
BFS kiểm chứng: 3


Bảng trạng thái cho bảng `5 × 5`:

| bước | giá trị | ý nghĩa |
|---|---|---|
| kiểm tra 4 góc | không vi phạm | mọi hàng là `10011` hoặc `01100` |
| `row_sum` | 3 ∈ [2, 3] | hàng 0 có 3 số 1 |
| `col_sum` | 2 ∈ [2, 3] | cột 0 có 2 số 1 |
| cột 0 | `1 0 1 0 0` so mẫu `1 0 1 0 1` | sai 1 chỗ (lẻ) → `5 - 1 = 4` → 2 lần đổi hàng |
| hàng 0 | `1 0 0 1 1` so mẫu `1 0 1 0 1` | sai 2 chỗ (chẵn) → 1 lần đổi cột |
| đáp án | `(4 + 2) / 2 = 3` | |

Vì sao cột 0 sai 1 chỗ lại phải dùng `5 - 1 = 4`? Cột 0 có hai số 1 và ba số 0 → đích phải là `0 1 0 1 0` (ba số 0), không thể là `1 0 1 0 1`. So với `01010`: sai ở vị trí 0, 1, 2, 3 → 4 chỗ → 2 lần đổi.

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def movesToChessboard(self, board: List[List[int]]) -> int:
        n = len(board)
        # 1) Bất biến: mọi hàng bằng hàng 0 hoặc là phần bù của hàng 0
        for i in range(n):
            for j in range(n):
                if board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j]:
                    return -1
        # 2) Cân bằng số 1 trên hàng 0 và cột 0
        row_sum = sum(board[0])
        col_sum = sum(board[i][0] for i in range(n))
        if not (n // 2 <= row_sum <= (n + 1) // 2):
            return -1
        if not (n // 2 <= col_sum <= (n + 1) // 2):
            return -1
        # 3) Đếm vị trí sai so với mẫu 1,0,1,0,...
        row_mis = sum(board[i][0] == i % 2 for i in range(n))   # quyết định đổi hàng
        col_mis = sum(board[0][j] == j % 2 for j in range(n))   # quyết định đổi cột
        if n % 2 == 1:
            if row_mis % 2:
                row_mis = n - row_mis
            if col_mis % 2:
                col_mis = n - col_mis
        else:
            row_mis = min(row_mis, n - row_mis)
            col_mis = min(col_mis, n - col_mis)
        return (row_mis + col_mis) // 2

for b, exp in EXAMPLES:
    assert Solution().movesToChessboard(b) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**(a) Điều kiện cần.** Đổi hàng/cột là các hoán vị. Quan hệ "hàng `i` bằng hàng `k`" và "hàng `i` là bù của hàng `k`" được bảo toàn (đổi cột tác động giống nhau lên mọi hàng). Trong bàn cờ, các hàng luân phiên `A, B, A, B, ...` với `B = bù(A)`. Vậy bảng ban đầu phải có mọi hàng ∈ {A, bù A} — tương đương điều kiện 4 góc. Số hàng loại A/B phải là `⌊n/2⌋` và `⌈n/2⌉`, mà loại của hàng `i` được đọc từ `board[i][0]` → `col_sum ∈ [n//2, (n+1)//2]`. Tương tự `row_sum` cho cột.

**(b) Điều kiện đủ + tối ưu.** Khi hai điều kiện thoả, ta có thể sắp hàng sao cho cột 0 xen kẽ, và sắp cột sao cho hàng 0 xen kẽ. Khi đó hàng 0 xen kẽ, mọi hàng khác là hàng 0 hoặc bù của nó, xen kẽ theo cột 0 → bàn cờ. Hai việc sắp độc lập vì thứ tự hàng không ảnh hưởng đến thứ tự cột và ngược lại.

**(c) Số lần đổi cho mỗi chiều.** Bài 1D: dãy 0/1 với số lượng mỗi loại cố định, đích là một dãy xen kẽ cùng số lượng. Số vị trí sai `w` luôn chẵn (`w/2` chỗ "thừa 1", `w/2` chỗ "thừa 0"). Mỗi lần đổi chỗ sửa nhiều nhất 2 vị trí sai, và luôn có cách đổi sửa đúng 2 (một chỗ thừa 1 với một chỗ thừa 0) → tối ưu là `w/2`.
- `n` chẵn: hai đích đều cùng số lượng → chọn đích ít sai hơn.
- `n` lẻ: chỉ đích có cùng số lượng 1 là hợp lệ; với đích sai, `w` sẽ lẻ (do số lượng lệch 1) → chọn `w` chẵn.

**(d) Tổng.** Số nước tối thiểu = tối thiểu cho hàng + tối thiểu cho cột, vì mỗi nước chỉ tác động lên một chiều.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS trạng thái | `O((n!)² n⁴)` | `O((n!)² n²)` | chỉ `n ≤ 4..5` |
| 2. Liệt kê hoán vị | `O((n!)² n²)` | `O(n²)` | cắt tỉa mạnh, vẫn giai thừa |
| 3. Bất biến + đếm | `O(n²)` | `O(1)` | **lời giải chuẩn** |
| 4. Bitmask | `O(n²)` | `O(n)` | gọn, phép bit nhanh |

Với `n = 30`, cách 3 chỉ ~900 phép so sánh.

In [17]:
import random, time

def random_valid_board(n, rnd):
    """Bắt đầu từ bàn cờ rồi hoán vị ngẫu nhiên hàng và cột -> luôn khả thi."""
    b = [[(i + j) % 2 for j in range(n)] for i in range(n)]
    P = list(range(n)); Q = list(range(n))
    rnd.shuffle(P); rnd.shuffle(Q)
    return [[b[P[i]][Q[j]] for j in range(n)] for i in range(n)]

rnd = random.Random(782)
big = random_valid_board(30, rnd)
for name, f in [("math", moves_math), ("bitmask", moves_bitmask), ("Solution", lambda b: Solution().movesToChessboard(b))]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:8}: {r}  ({(time.perf_counter() - t0) * 1000:.2f} ms)")

math    : 11  (0.09 ms)
bitmask : 11  (0.14 ms)
Solution: 11  (0.08 ms)


## 12. Bộ test

In [18]:
methods = {
    "bfs": moves_bfs,
    "perm": moves_perm,
    "math": moves_math,
    "bitmask": moves_bitmask,
    "SolutionMath": lambda b: SolutionMath().movesToChessboard(b),
    "SolutionBitmask": lambda b: SolutionBitmask().movesToChessboard(b),
    "Solution": lambda b: Solution().movesToChessboard(b),
}
fast = ["math", "bitmask", "SolutionMath", "SolutionBitmask", "Solution"]

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([[1, 0], [0, 1]], 0),
    ([[0, 0], [1, 1]], -1),                   # hàng hằng: không cân bằng
    ([[0, 1], [0, 1]], -1),
    ([[1, 1, 0], [0, 0, 1], [0, 0, 1]], 2),   # n lẻ
    ([[0, 0, 1], [0, 0, 1], [1, 1, 0]], 2),
    ([[1, 1, 1], [0, 0, 0], [1, 1, 1]], -1),  # hàng không cân bằng
    ([[1, 0, 1], [0, 1, 0], [1, 0, 1]], 0),
    ([[0, 1, 0], [1, 0, 1], [0, 1, 0]], 0),
]
for name, f in methods.items():
    for b, exp in edge:
        assert f(b) == exp, (name, b, f(b), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên hoàn toàn (đa số -1), n = 2..4, đối chiếu BFS
for seed in range(400):
    r = random.Random(seed)
    n = r.randint(2, 4)
    b = [[r.randint(0, 1) for _ in range(n)] for _ in range(n)]
    ref = moves_bfs(b)
    for name, f in methods.items():
        assert f(b) == ref, (name, b)
print("400 bảng ngẫu nhiên: khớp BFS")

# 12.3 Bảng khả thi (hoán vị từ bàn cờ), n = 2..5, đối chiếu BFS
for seed in range(150):
    r = random.Random(1000 + seed)
    n = r.randint(2, 5)
    b = random_valid_board(n, r)
    if r.random() < 0.5:
        b = [[1 - x for x in row] for row in b]     # đảo màu
    ref = moves_bfs(b)
    assert ref >= 0
    for name, f in methods.items():
        if name == "perm" and n == 5 and seed % 5:  # perm n=5 chậm hơn, kiểm tra 1/5 số lần
            continue
        assert f(b) == ref, (name, b)
print("150 bảng khả thi: khớp BFS")

# 12.4 n lớn: các cách nhanh khớp nhau; thêm nhiễu một ô -> -1
for seed in range(50):
    r = random.Random(5000 + seed)
    n = r.randint(6, 30)
    b = random_valid_board(n, r)
    vals = {name: methods[name](b) for name in fast}
    assert len(set(vals.values())) == 1 and vals["Solution"] >= 0, vals
    i, j = r.randrange(n), r.randrange(n)
    b[i][j] ^= 1
    assert all(methods[name](b) == -1 for name in fast)
print("n lớn: OK")

Ví dụ + edge cases: OK


400 bảng ngẫu nhiên: khớp BFS
150 bảng khả thi: khớp BFS


n lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Luôn lấy `min(mis, n - mis)` kể cả khi `n` lẻ

**Code sai:**

In [19]:
def wrong_odd(board):
    n = len(board)
    if any(board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] for i in range(n) for j in range(n)):
        return -1
    rs, cs = sum(board[0]), sum(board[i][0] for i in range(n))
    if not (n // 2 <= rs <= (n + 1) // 2 and n // 2 <= cs <= (n + 1) // 2):
        return -1
    rm = sum(board[i][0] == i % 2 for i in range(n))
    cm = sum(board[0][j] == j % 2 for j in range(n))
    rm, cm = min(rm, n - rm), min(cm, n - cm)        # SAI khi n lẻ
    return (rm + cm) // 2

print("sai:", wrong_odd(b5), " đúng:", Solution().movesToChessboard(b5))

sai: 1  đúng: 3


**Vì sao sai:** khi `n` lẻ, chỉ một mẫu xen kẽ có số lượng 1 khớp. Mẫu còn lại cho số sai **lẻ** — chia 2 bị làm tròn xuống, cho kết quả nhỏ hơn thực tế (không thể đạt). Với `b5`, `row_mis = 1` → sai lấy `min(1, 4) = 1` → `1 // 2 = 0` lần đổi hàng (không thể!).

**Code đúng:** `n` lẻ → nếu số sai lẻ thì thay bằng `n - mis`.

### Lỗi 2 — Chỉ kiểm tra mỗi hàng có đúng số lượng 1, quên kiểm tra quan hệ giữa các hàng

**Code sai:**

In [20]:
def wrong_count_only(board):
    n = len(board)
    for r in board:
        if not (n // 2 <= sum(r) <= (n + 1) // 2):
            return -1
    for j in range(n):
        if not (n // 2 <= sum(board[i][j] for i in range(n)) <= (n + 1) // 2):
            return -1
    rm = sum(board[i][0] == i % 2 for i in range(n))
    cm = sum(board[0][j] == j % 2 for j in range(n))
    if n % 2:
        rm = n - rm if rm % 2 else rm
        cm = n - cm if cm % 2 else cm
    else:
        rm, cm = min(rm, n - rm), min(cm, n - cm)
    return (rm + cm) // 2

bad = [[0, 1, 1, 0], [1, 0, 0, 1], [1, 1, 0, 0], [0, 0, 1, 1]]
print("sai:", wrong_count_only(bad), " đúng:", Solution().movesToChessboard(bad), " BFS:", moves_bfs(bad))

sai: 2  đúng: -1  BFS: -1


**Vì sao sai:** mọi hàng/cột đều có hai số 1, nhưng hàng `1100` không phải `0110` cũng không phải bù của nó → không bao giờ thành bàn cờ. Số lượng là điều kiện cần, chưa đủ.

**Code đúng:** thêm kiểm tra 4 góc (hoặc kiểm tra "chỉ có hai loại hàng bù nhau").

### Lỗi 3 — Đếm sai trên chiều nhầm (dùng hàng 0 để tính số lần đổi hàng)

**Code sai:**

In [21]:
def wrong_axis(board):
    n = len(board)
    if any(board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] for i in range(n) for j in range(n)):
        return -1
    rs, cs = sum(board[0]), sum(board[i][0] for i in range(n))
    if not (n // 2 <= rs <= (n + 1) // 2 and n // 2 <= cs <= (n + 1) // 2):
        return -1
    # SAI: tính "đổi hàng" từ hàng 0 và nhân đôi
    m = sum(board[0][j] == j % 2 for j in range(n))
    m = min(m, n - m) if n % 2 == 0 else (n - m if m % 2 else m)
    return m          # coi như hàng và cột cần cùng số lần

b_test = [[0, 0, 1, 1], [1, 1, 0, 0], [0, 0, 1, 1], [1, 1, 0, 0]]
print("sai:", wrong_axis(b_test), " đúng:", Solution().movesToChessboard(b_test))

sai: 2  đúng: 1


**Vì sao sai:** số lần đổi hàng do **cột 0** quyết định (loại của từng hàng), số lần đổi cột do **hàng 0** quyết định. Hai con số này nói chung khác nhau; ở `b_test` các hàng đã xen kẽ (0 lần) nhưng các cột cần 1 lần.

**Code đúng:** `row_mis` từ `board[i][0]`, `col_mis` từ `board[0][j]`, cộng lại rồi chia 2.

## 14. Mẹo & nhận dạng pattern

- **"Được đổi chỗ hàng/cột tuỳ ý"** → nghĩ ngay tới **bất biến**: nội dung từng hàng (như multiset), quan hệ bằng/bù giữa các hàng, số lượng phần tử.
- **Tách chiều độc lập:** khi các phép toán trên hàng không ảnh hưởng thứ tự cột (và ngược lại), đáp án = tổng hai bài 1D.
- **"Số lần đổi chỗ tối thiểu với hai loại phần tử"** = số vị trí sai / 2. Với phần tử phân biệt: `n - số chu trình`.
- **Chẵn/lẻ:** mỗi khi có "hai mẫu đích", kiểm tra xem cả hai có hợp lệ không (đặc biệt khi `n` lẻ).
- Điều kiện 4 góc `a[0][0]^a[i][0]^a[0][j]^a[i][j] == 0` là cách nhanh để kiểm tra "ma trận có hạng 1 trên GF(2) cộng hằng" — hay xuất hiện trong bài lật hàng/cột.

## 15. Biến thể

1. **Chỉ được đổi hàng:** cần mọi **cột** đã xen kẽ đúng... cụ thể: hàng 0 phải xen kẽ sẵn, rồi đáp án = `row_mis` (đã chỉnh) / 2.
2. **Được lật (flip) hàng/cột thay vì đổi chỗ** (LeetCode 2128 *Remove All Ones With Row and Column Flips*): điều kiện mọi hàng bằng hoặc bù hàng 0.
3. **Đổi chỗ chỉ các hàng kề nhau:** số lần đổi = số nghịch thế (bubble sort) thay vì `sai/2`.
4. **Bảng `m × n` chữ nhật:** cùng ý tưởng; điều kiện cân bằng dùng `m` cho cột 0 và `n` cho hàng 0.
5. **Đích là một mẫu khác (không phải bàn cờ):** quy về so khớp loại hàng/cột với mẫu.

Minh hoạ biến thể 4 (bảng chữ nhật):

In [22]:
def moves_rect(board):
    m, n = len(board), len(board[0])
    if any(board[0][0] ^ board[i][0] ^ board[0][j] ^ board[i][j] for i in range(m) for j in range(n)):
        return -1
    def cost(seq):
        L = len(seq)
        ones = sum(seq)
        if not (L // 2 <= ones <= (L + 1) // 2):
            return None
        mis = sum(seq[i] == i % 2 for i in range(L))
        if L % 2:
            mis = L - mis if mis % 2 else mis
        else:
            mis = min(mis, L - mis)
        return mis // 2
    a = cost([board[i][0] for i in range(m)])
    b = cost(board[0])
    return -1 if a is None or b is None else a + b

print(moves_rect([[0, 1, 1], [1, 0, 0]]), moves_rect([[0, 0, 1, 1], [1, 1, 0, 0], [0, 0, 1, 1]]))

1 1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | duyệt mọi bảng tới được | `O((n!)² n⁴)` | lớn | chuẩn đối chiếu, `n ≤ 4` |
| 2. Liệt kê hoán vị | hoán vị hàng × cột, chi phí = n − chu trình | `O((n!)² n²)` | `O(n²)` | đối chiếu, minh hoạ tính độc lập |
| 3. Bất biến + đếm sai | kiểm tra 4 góc, cân bằng, sai/2 | `O(n²)` | `O(1)` | **nộp bài** |
| 4. Bitmask + Counter | hai loại hàng bù nhau, popcount | `O(n²)` | `O(n)` | khi thích phép bit, dễ mở rộng |

## 17. Bài tập

**Bài 1 (đoán output).** `movesToChessboard([[1,1,0,0],[0,0,1,1],[1,1,0,0],[0,0,1,1]])` và `movesToChessboard([[1,0,0],[0,1,1],[1,0,0]])` = ?

**Bài 2 (tính tay).** Với dãy `0 0 0 1 1` (n = 5), tính số lần đổi tối thiểu để thành dãy xen kẽ. Dãy đích là gì?

**Bài 3 (điền TODO).** Hoàn thiện `line_swaps(seq)` trả về số lần đổi tối thiểu để dãy 0/1 `seq` thành xen kẽ, hoặc `-1`.

**Bài 4 (tự viết lại).** Viết lại lời giải sao cho **trả về luôn danh sách các nước đi** (dạng `('R', i, j)` hoặc `('C', i, j)`), và kiểm tra áp dụng chúng thực sự tạo ra bàn cờ.

**Bài 5 (mini challenge).** Biến thể "chỉ được đổi chỗ hai hàng **kề nhau** hoặc hai cột **kề nhau**". Viết `moves_adjacent(board)`.

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def line_swaps(seq):
    # TODO: kiểm tra cân bằng; đếm sai so với mẫu 1010..; xử lý chẵn/lẻ
    pass

# Bài 4 — TODO
def moves_with_plan(board):
    # TODO: với mỗi chiều, chọn mẫu đích; ghép các vị trí "thừa 1" với "thừa 0" thành cặp đổi
    pass

# Bài 5 — TODO
def moves_adjacent(board):
    # TODO: với mỗi chiều và mỗi mẫu hợp lệ, số lần đổi kề = tổng |vị trí số 1 thứ k - vị trí đích thứ k|
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: kiểm tra mọi hàng là `1100` hoặc `0011` → khả thi.
- Mức 2: các hàng đã xen kẽ; hàng 0 `1100` cần 1 lần đổi cột.
- Mức 3: bảng 3×3: hàng `100` có một số 1, cột 0 `101` có hai số 1 — kiểm tra 4 góc và cân bằng.

**Bài 2.**
- Mức 1: có hai số 1, ba số 0 → đích `01010`.
- Mức 2: so sánh `00011` với `01010`: sai ở vị trí 1 và 4.
- Mức 3: 2 chỗ sai → 1 lần đổi.

**Bài 3.**
- Mức 1: `ones = sum(seq)`, phải thuộc `[L//2, (L+1)//2]`.
- Mức 2: `mis = sum(seq[i] == i % 2)`.
- Mức 3: chẵn → `min`; lẻ → chọn số chẵn; chia 2.

**Bài 4.**
- Mức 1: chọn mẫu đích như trong lời giải.
- Mức 2: danh sách `A` = vị trí đáng là 0 nhưng là 1; `B` = ngược lại; `|A| == |B|`.
- Mức 3: ghép `A[k]` với `B[k]`; áp dụng đổi hàng trước rồi đổi cột, kiểm tra `is_chessboard`.

**Bài 5.**
- Mức 1: hai chiều vẫn độc lập.
- Mức 2: đổi kề = đếm nghịch thế; với dãy 0/1, bằng tổng khoảng cách ghép số 1 thứ `k` với vị trí 1 thứ `k` của đích.
- Mức 3: thử cả hai mẫu (nếu hợp lệ), lấy min.

## 19. Đáp án tham khảo

In [24]:
S = Solution()
# Bài 1
assert S.movesToChessboard([[1, 1, 0, 0], [0, 0, 1, 1], [1, 1, 0, 0], [0, 0, 1, 1]]) == 1
assert S.movesToChessboard([[1, 0, 0], [0, 1, 1], [1, 0, 0]]) == 1
assert moves_bfs([[1, 0, 0], [0, 1, 1], [1, 0, 0]]) == 1

# Bài 2 + Bài 3
def line_swaps_ans(seq):
    L = len(seq)
    ones = sum(seq)
    if not (L // 2 <= ones <= (L + 1) // 2):
        return -1
    mis = sum(seq[i] == i % 2 for i in range(L))
    if L % 2:
        if mis % 2:
            mis = L - mis
    else:
        mis = min(mis, L - mis)
    return mis // 2

def line_swaps_brute(seq):
    L = len(seq)
    best = -1
    for start in (0, 1):
        tgt = [(start + i) % 2 for i in range(L)]
        if sorted(tgt) == sorted(seq):
            c = sum(a != b for a, b in zip(seq, tgt)) // 2
            best = c if best < 0 else min(best, c)
    return best

assert line_swaps_ans([0, 0, 0, 1, 1]) == 1
for L in range(1, 9):
    for mask in range(1 << L):
        seq = [(mask >> i) & 1 for i in range(L)]
        assert line_swaps_ans(seq) == line_swaps_brute(seq)

# Bài 4
def moves_with_plan_ans(board):
    n = len(board)
    if S.movesToChessboard(board) < 0:
        return None
    def plan(seq):
        best = None
        for start in (0, 1):
            tgt = [(start + i) % 2 for i in range(n)]
            if sum(tgt) != sum(seq):
                continue
            A = [i for i in range(n) if seq[i] == 1 and tgt[i] == 0]
            B = [i for i in range(n) if seq[i] == 0 and tgt[i] == 1]
            pairs = list(zip(A, B))
            if best is None or len(pairs) < len(best):
                best = pairs
        return best
    moves = [('R', i, j) for i, j in plan([board[i][0] for i in range(n)])]
    moves += [('C', i, j) for i, j in plan(board[0])]
    return moves

for seed in range(200):
    r = random.Random(9000 + seed)
    n = r.randint(2, 8)
    b = random_valid_board(n, r)
    mv = moves_with_plan_ans(b)
    assert len(mv) == S.movesToChessboard(b)
    cur = b
    for kind, i, j in mv:
        cur = swap_rows(cur, i, j) if kind == 'R' else swap_cols(cur, i, j)
    assert is_chessboard(cur)

# Bài 5
def moves_adjacent_ans(board):
    n = len(board)
    if S.movesToChessboard(board) < 0:
        return -1
    def adj_cost(seq):
        ones = [i for i in range(n) if seq[i] == 1]
        best = None
        for start in (0, 1):
            tgt = [i for i in range(n) if (start + i) % 2 == 1]
            if len(tgt) != len(ones):
                continue
            c = sum(abs(a - b) for a, b in zip(ones, tgt))
            best = c if best is None else min(best, c)
        return best
    return adj_cost([board[i][0] for i in range(n)]) + adj_cost(board[0])

def moves_adjacent_bfs(board):
    n = len(board)
    start = to_key(board)
    dist = {start: 0}
    q = deque([start])
    while q:
        s = q.popleft()
        if is_chessboard(s):
            return dist[s]
        rows = list(s)
        for i in range(n - 1):
            r2 = rows[:]; r2[i], r2[i + 1] = r2[i + 1], r2[i]
            t = tuple(r2)
            if t not in dist:
                dist[t] = dist[s] + 1; q.append(t)
            t = tuple(tuple(r[i + 1] if k == i else r[i] if k == i + 1 else r[k] for k in range(n)) for r in rows)
            if t not in dist:
                dist[t] = dist[s] + 1; q.append(t)
    return -1

for seed in range(120):
    r = random.Random(7000 + seed)
    n = r.randint(2, 4)
    b = random_valid_board(n, r) if seed % 3 else [[r.randint(0, 1) for _ in range(n)] for _ in range(n)]
    assert moves_adjacent_ans(b) == moves_adjacent_bfs(b), b
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
n = len(b)
if any(b[0][0]^b[i][0]^b[0][j]^b[i][j] for i in range(n) for j in range(n)): return -1
rs, cs = sum(b[0]), sum(b[i][0] for i in range(n))
if not (n//2 <= rs <= (n+1)//2 and n//2 <= cs <= (n+1)//2): return -1
rm = sum(b[i][0] == i % 2 for i in range(n))   # đổi hàng
cm = sum(b[0][j] == j % 2 for j in range(n))   # đổi cột
if n % 2: rm = n - rm if rm % 2 else rm; cm = n - cm if cm % 2 else cm
else:     rm, cm = min(rm, n - rm), min(cm, n - cm)
return (rm + cm) // 2
```

**Checklist trước khi submit:**

- [ ] Kiểm tra điều kiện 4 góc cho **mọi** `(i, j)`.
- [ ] Kiểm tra cân bằng số 1 cho **cả** hàng 0 và cột 0.
- [ ] `row_mis` lấy từ **cột 0**, `col_mis` lấy từ **hàng 0**.
- [ ] `n` lẻ: chọn số sai **chẵn**; `n` chẵn: lấy `min`.
- [ ] Chia 2 **sau khi** cộng (hoặc chia từng phần — đều chẵn nên như nhau).